# TP 1 — PCA et LDA

**Objectifs**

- Visualiser le dataset `digits` (64 dimensions) en 2D via PCA et via LDA.
- Tracer la courbe de variance expliquée pour choisir un nombre de composantes.
- Mesurer le gain d'une réduction PCA avant un classifieur logistique.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(X.shape)

(1797, 64)


## Exercice 1 — Projection PCA en 2D

1. Standardisez les données (`StandardScaler`) fitté sur `X_train`.
2. Projetez en 2D via `PCA(n_components=2)`.
3. Tracez un scatter coloré par classe (utilisez `cmap="tab10"`).
4. Affichez la variance expliquée totale des 2 premières composantes.

<details>
<summary>💡 Coup de pouce — PCA pour projection 2D</summary>

**🎯 But :** projeter un dataset haute dimension en 2D pour visualiser les structures (clusters, séparabilité).

**Toujours standardiser avant PCA**

```python
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler().fit(X_train)         # fit UNIQUEMENT sur train
X_train_std = scaler.transform(X_train)
X_test_std  = scaler.transform(X_test)         # même scaler pour test
```

⚠️ **Erreur classique = leakage** : si vous fittez le scaler sur `X` complet (train + test), vous utilisez de l'info du test pour transformer le train → résultats trompeurs en évaluation.

**Calculer et appliquer la PCA**

```python
pca = PCA(n_components=2).fit(X_train_std)
X_proj = pca.transform(X_train_std)            # shape (n_samples, 2)
```

Ou en raccourci si on ne réutilise pas la PCA ailleurs :

```python
X_proj = PCA(n_components=2).fit_transform(X_train_std)
```

**Visualiser**

```python
plt.scatter(X_proj[:, 0], X_proj[:, 1], c=y_train, cmap='tab10', alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.colorbar()
```

`c=y_train` colore chaque point selon sa classe. `cmap='tab10'` donne 10 couleurs distinctes (parfait pour Digits).

**Mesurer la perte d'info**

```python
print(f"Variance expliquée : {pca.explained_variance_ratio_}")
print(f"Total : {pca.explained_variance_ratio_.sum():.1%}")
```

Sur Digits 64D → 2D, vous garderez typiquement **30-40 %** de variance. C'est peu, mais souvent suffisant pour voir les groupes principaux. Pour la perf classif, il faudra plus de composantes (vu en Exo 4).

</details>

In [2]:
# TODO


## Exercice 2 — Variance cumulée

Fit `PCA()` sans arg sur `X_train` standardisé. Tracez la variance expliquée cumulée en fonction du nombre de composantes. Quel `n_components` faut-il pour garder 90 % de la variance ? 95 % ?

<details>
<summary>💡 Coup de pouce — variance expliquée cumulée</summary>

**🎯 But :** trouver le nombre minimum de composantes principales pour conserver, par exemple, 90 % de l'information.

**Toutes les composantes d'un coup**

```python
pca_full = PCA().fit(X_train_std)              # sans n_components → toutes
```

Le nombre de composantes est `min(n_samples, n_features)`. Sur Digits : 64.

**Variance expliquée cumulée**

```python
import numpy as np
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
```

`cumvar` est un tableau croissant de 64 valeurs entre 0 et 1, qui finit à 1.0.

**Tracer**

```python
plt.plot(range(1, len(cumvar) + 1), cumvar, marker='.')
plt.axhline(0.9, color='red', linestyle='--', label='90 %')
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance cumulée')
plt.legend()
```

Vous verrez une courbe qui monte vite puis sature : c'est typique des données avec corrélations entre features. Plus la courbe est **incurvée**, plus la PCA est efficace.

**Trouver le nombre de composantes pour 90 %**

```python
n_90 = np.argmax(cumvar >= 0.90) + 1
print(f"Pour 90 % de variance : {n_90} composantes")
```

Décortique :
- `cumvar >= 0.90` : tableau booléen, True à partir du 1er indice atteignant 0.90.
- `np.argmax` sur un booléen renvoie l'indice du **premier True**.
- `+ 1` car les indices NumPy sont 0-based mais on compte les composantes 1-based.

Sur Digits, attendez-vous à **~20-25 composantes** pour 90 %. → On peut diviser la dimensionnalité par ~3 sans perdre beaucoup.

</details>

In [3]:
# TODO


## Exercice 3 — LDA en 2D

1. Projetez `X_train` (standardisé) en 2D avec `LinearDiscriminantAnalysis(n_components=2)`. Attention : LDA est **supervisée** → il faut passer `y_train`.
2. Affichez le scatter coloré par classe.
3. Comparez visuellement avec la projection PCA précédente. Quelle projection sépare mieux les classes ?

<details>
<summary>💡 Coup de pouce — LDA, l'alternative supervisée à PCA</summary>

**🎯 But :** projeter en basse dimension en **utilisant l'info des classes** pour maximiser leur séparabilité.

**PCA vs LDA — la différence essentielle**

| | PCA | LDA |
|---|---|---|
| Supervisée ? | ❌ ignore `y` | ✅ utilise `y` |
| Critère | Variance maximale | Séparation des classes maximale |
| Limite de composantes | min(n_samples, n_features) | min(n_features, **n_classes − 1**) |

**Construire la LDA**

```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = LinearDiscriminantAnalysis(n_components=2).fit(X_train_std, y_train)
X_lda = lda.transform(X_train_std)
```

⚠️ **Erreur classique** : oublier `y_train` dans `.fit()`. La LDA plantera ou donnera n'importe quoi. C'est obligatoire.

**Pourquoi la limite `n_classes − 1` ?**

Mathématiquement, la LDA cherche des projections qui séparent les classes. Avec K classes, on peut avoir au maximum **K − 1 directions** discriminantes indépendantes (intuition : 2 classes → 1 axe, 3 classes → 1 plan, etc.).

Sur Digits : 10 classes → max **9 composantes LDA**. Pour une visualisation 2D, OK avec n_components=2.

**Visualisation comparative**

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_proj[:, 0], X_proj[:, 1], c=y_train, cmap='tab10', alpha=0.7)
axes[0].set_title('PCA (2D)')
axes[1].scatter(X_lda[:, 0], X_lda[:, 1], c=y_train, cmap='tab10', alpha=0.7)
axes[1].set_title('LDA (2D)')
```

Visuellement, la **LDA sépare presque toujours mieux** les classes que la PCA — c'est son objectif explicite.

**Quand préférer l'un à l'autre**

- **PCA** : exploration sans labels, compression générique, débruitage.
- **LDA** : classification linéaire native (LDA est aussi un classifieur), visualisation supervisée, datasets fortement structurés en classes.

</details>

In [4]:
# TODO


## Exercice 4 — Effet de PCA sur la classification

Comparez trois Pipelines via `cross_val_score(cv=5)` sur le train :

- A : `StandardScaler + LogisticRegression(max_iter=2000)` (sans réduction)
- B : `StandardScaler + PCA(n_components=10) + LogisticRegression(max_iter=2000)`
- C : `StandardScaler + PCA(n_components=30) + LogisticRegression(max_iter=2000)`

Affichez moyenne ± std pour chacun. Commentez : la réduction de dimension dégrade-t-elle ou améliore-t-elle la performance ?

<details>
<summary>💡 Coup de pouce — effet de PCA sur la classification</summary>

**🎯 But :** comparer deux pipelines (avec ou sans PCA) pour voir si la réduction de dimension aide ou nuit à la classification.

**Pipeline A — sans PCA**

```python
pipe_a = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=2000))
])
```

**Pipeline B — avec PCA**

```python
pipe_b = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=10)),
    ('clf',    LogisticRegression(max_iter=2000))
])
```

L'ordre `scaler → pca → clf` est important : la PCA ne fonctionne bien que sur des données standardisées (sinon les features de grande échelle dominent).

**Comparer en CV**

```python
from sklearn.model_selection import cross_val_score
scores_a = cross_val_score(pipe_a, X_train, y_train, cv=5, scoring='accuracy')
scores_b = cross_val_score(pipe_b, X_train, y_train, cv=5, scoring='accuracy')
print(f"Sans PCA : {scores_a.mean():.3f} ± {scores_a.std():.3f}")
print(f"Avec PCA : {scores_b.mean():.3f} ± {scores_b.std():.3f}")
```

**Ce que vous devriez observer**

Sur Digits :
- Pipeline A : accuracy ≈ 0.96
- Pipeline B (n=10) : accuracy ≈ 0.94-0.97

Souvent **B est proche ou meilleur** que A, alors qu'on a divisé la dim par 6 ! Pourquoi :
- PCA agit comme une **régularisation** : on supprime les composantes faibles qui pouvaient porter du bruit.
- Le classifieur a moins de paramètres à apprendre → moins d'overfit.

**Quand PCA aide vs nuit**

| Situation | Effet de PCA |
|---|---|
| Beaucoup de features bruitées | ✅ Aide (régularisation) |
| Peu d'échantillons, beaucoup de features | ✅ Aide (réduit overfitting) |
| Features déjà toutes pertinentes | ⚠️ Neutre ou nuit |
| Très peu de composantes (n=2) | ❌ Perd de l'info, dégrade |

</details>

In [5]:
# TODO
